# device check

In [36]:
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


# path

In [37]:
DATA_ROOT = Path(r"D:\dataset\sampled_500")

TRAIN_DIR = DATA_ROOT / "train_mini"
VAL_DIR = DATA_ROOT / "validation"

MODEL_DIR = DATA_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

MODEL_PATH = MODEL_DIR / "inat_resnet50_best.pth"

#processing

In [38]:
INPUT_SIZE = 320
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        350,
        scale=(0.80, 1.0),       # at lease 80%
        ratio=(0.85, 1.15),      # ratio limit
        interpolation=transforms.InterpolationMode.BILINEAR,
        antialias=True
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.10
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize(352, interpolation=transforms.InterpolationMode.BILINEAR,
        antialias=True),
    transforms.CenterCrop(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# load traning and validation

In [39]:
train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    VAL_DIR,
    transform=val_transform
)

assert train_dataset.class_to_idx == val_dataset.class_to_idx, (
    "Train and validation folder differed"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=8,  
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=8,
    pin_memory=torch.cuda.is_available()
)

print("Classes:", len(train_dataset.classes))
print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Class mapping:", train_dataset.class_to_idx)

Classes: 500
Train images: 20000
Validation images: 5000
Class mapping: {'00001_Animalia_Annelida_Polychaeta_Sabellida_Sabellidae_Sabella_spallanzanii': 0, '00003_Animalia_Annelida_Polychaeta_Sabellida_Serpulidae_Spirobranchus_cariniferus': 1, '00018_Animalia_Arthropoda_Arachnida_Araneae_Araneidae_Argiope_bruennichi': 2, '00024_Animalia_Arthropoda_Arachnida_Araneae_Araneidae_Cyclosa_turbinata': 3, '00038_Animalia_Arthropoda_Arachnida_Araneae_Araneidae_Neoscona_crucifera': 4, '00055_Animalia_Arthropoda_Arachnida_Araneae_Filistatidae_Kukulcania_hibernalis': 5, '00128_Animalia_Arthropoda_Arachnida_Araneae_Thomisidae_Synema_globosum': 6, '00145_Animalia_Arthropoda_Arachnida_Opiliones_Phalangiidae_Phalangium_opilio': 7, '00159_Animalia_Arthropoda_Chilopoda_Scolopendromorpha_Scolopendridae_Scolopendra_heros': 8, '00203_Animalia_Arthropoda_Insecta_Coleoptera_Carabidae_Cicindela_hirticollis': 9, '00216_Animalia_Arthropoda_Insecta_Coleoptera_Carabidae_Scaphinotus_angusticollis': 10, '00230_Anim

# training function

In [40]:
def training(EPOCHS,model,optimizer,scheduler,criterion,fname):
    best_val_accuracy = 0.0

    early_stop_patience = 7
    epochs_without_improvement = 0
    min_improvement = 1e-5

    for epoch in range(EPOCHS):
        # Training
        model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            train_correct += (
                outputs.argmax(dim=1) == labels
            ).sum().item()
            train_total += labels.size(0)

        # Validation
        model.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                val_correct += (
                    outputs.argmax(dim=1) == labels
                ).sum().item()
                val_total += labels.size(0)

        # result
        train_accuracy = train_correct / train_total
        val_accuracy = val_correct / val_total

        scheduler.step(val_accuracy)

        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | "
            f"Train loss: {train_loss / train_total:.4f} | "
            f"Train acc: {train_accuracy:.4f} | "
            f"Val loss: {val_loss / val_total:.4f} | "
            f"Val acc: {val_accuracy:.4f}"
        )

        num_classes = len(train_dataset.classes)
        model_filename = fname
        # model_filename = f"inat_resnet18_{num_classes}classes_imagenet.pth"

        if val_accuracy > best_val_accuracy + min_improvement:
            best_val_accuracy = val_accuracy
            epochs_without_improvement = 0

            torch.save({
                "model_state_dict": model.state_dict(),
                "classes": train_dataset.classes,
                "num_classes": num_classes
            }, model_filename)

            print(f"Model saved as: {model_filename}")
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= early_stop_patience:
            print("Early stopping triggered.")
            break

# load pretrained model

In [41]:
weights = ResNet18_Weights.IMAGENET1K_V1

pt_model = resnet18(weights=weights)

num_classes = len(train_dataset.classes)

in_features = pt_model.fc.in_features





pt_model.fc = nn.Linear(
    pt_model.fc.in_features,
    num_classes
)


pt_model = pt_model.to(device)

pt_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

pt_optimizer = torch.optim.AdamW(
    pt_model.parameters(),
    lr=1e-5,
    weight_decay=5e-4
)

pt_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    pt_optimizer,
    mode="max",       # moniotr val accuracy
    factor=0.3,       # lr * 0.3
    patience=2,       # if no increase drop lr
    threshold=1e-4,
    min_lr=1e-7
)

num_classes = len(train_dataset.classes)
pt_model_filename = f"inat_resnet18_{num_classes}_ptclasses_imagenet.pth"

training(50,pt_model,pt_optimizer,pt_scheduler,pt_criterion,pt_model_filename)


Epoch 01/50 | Train loss: 6.1665 | Train acc: 0.0101 | Val loss: 5.8890 | Val acc: 0.0346
Model saved as: inat_resnet18_500_ptclasses_imagenet.pth
Epoch 02/50 | Train loss: 5.7087 | Train acc: 0.0765 | Val loss: 5.4543 | Val acc: 0.1254
Model saved as: inat_resnet18_500_ptclasses_imagenet.pth
Epoch 03/50 | Train loss: 5.3358 | Train acc: 0.1685 | Val loss: 5.1042 | Val acc: 0.2028
Model saved as: inat_resnet18_500_ptclasses_imagenet.pth
Epoch 04/50 | Train loss: 5.0284 | Train acc: 0.2504 | Val loss: 4.7994 | Val acc: 0.2662
Model saved as: inat_resnet18_500_ptclasses_imagenet.pth
Epoch 05/50 | Train loss: 4.7562 | Train acc: 0.3114 | Val loss: 4.5296 | Val acc: 0.3120
Model saved as: inat_resnet18_500_ptclasses_imagenet.pth
Epoch 06/50 | Train loss: 4.5180 | Train acc: 0.3587 | Val loss: 4.3155 | Val acc: 0.3486
Model saved as: inat_resnet18_500_ptclasses_imagenet.pth
Epoch 07/50 | Train loss: 4.3000 | Train acc: 0.4058 | Val loss: 4.1188 | Val acc: 0.3796
Model saved as: inat_resnet1

KeyboardInterrupt: 

# train with no preset weights

In [ ]:

np_model = resnet18(weights=None)

num_classes = len(train_dataset.classes)

np_model.fc = nn.Linear(
    np_model.fc.in_features,
    num_classes
)

np_model = np_model.to(device)

np_criterion = nn.CrossEntropyLoss(label_smoothing=0.2)

np_optimizer = torch.optim.AdamW(
    np_model.parameters(),
    lr=1e-4,
    weight_decay=5e-4
)

np_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    np_optimizer,
    mode="max",       # moniotr val accuracy
    factor=0.3,       # lr * 0.3
    patience=2,       # if no increase drop lr
    threshold=1e-3,
    min_lr=1e-7
)

num_classes = len(train_dataset.classes)
np_model_filename = f"inat_resnet18_{num_classes}_npclasses_imagenet.pth"

training(50,np_model,np_optimizer,np_scheduler,np_criterion,np_model_filename)

Epoch 01/50 | Train loss: 2.9224 | Train acc: 0.1200 | Val loss: 2.9320 | Val acc: 0.0900
Model saved as: inat_resnet18_20_npclasses_imagenet.pth
Epoch 02/50 | Train loss: 2.6426 | Train acc: 0.2637 | Val loss: 2.6845 | Val acc: 0.2300
Model saved as: inat_resnet18_20_npclasses_imagenet.pth
Epoch 03/50 | Train loss: 2.5196 | Train acc: 0.3175 | Val loss: 2.6876 | Val acc: 0.2150
Epoch 04/50 | Train loss: 2.4453 | Train acc: 0.3613 | Val loss: 2.5580 | Val acc: 0.2900
Model saved as: inat_resnet18_20_npclasses_imagenet.pth
Epoch 05/50 | Train loss: 2.3789 | Train acc: 0.3937 | Val loss: 2.6122 | Val acc: 0.2750
Epoch 06/50 | Train loss: 2.2984 | Train acc: 0.4138 | Val loss: 2.6202 | Val acc: 0.2600
Epoch 07/50 | Train loss: 2.2350 | Train acc: 0.4825 | Val loss: 2.6173 | Val acc: 0.2800
Epoch 08/50 | Train loss: 2.1187 | Train acc: 0.5350 | Val loss: 2.3808 | Val acc: 0.3850
Model saved as: inat_resnet18_20_npclasses_imagenet.pth
Epoch 09/50 | Train loss: 2.0617 | Train acc: 0.5750 | V

KeyboardInterrupt: 

# model reloading

In [ ]:
# CHECKPOINT_PATH= Path("inat_resnet18_500_ptclasses_imagenet.pth")
# RESUME_EPOCH_IF_MISSING=21
# RESUME_BEST_ACC_IF_MISSING=0.60

# num_classes = len(train_dataset.classes)

# checkpoint = torch.load(
#         CHECKPOINT_PATH,
#         map_location="cpu"
#     )

# checkpoint_classes = checkpoint.get(
#     "num_classes",
#     num_classes
# )

# if checkpoint_classes != num_classes:
#     raise ValueError(
#         f"Checkpoint has {checkpoint_classes} classes, "
#         f"but dataset has {num_classes} classes."
#     )

# if "classes" in checkpoint:
#     if list(checkpoint["classes"]) != list(train_dataset.classes):
#         raise ValueError(
#             "Checkpoint class order does not match "
#             "the current dataset class order."
#         )

# rs_model = resnet18(weights=None)

# in_features = rs_model.fc.in_features

# rs_model.fc = nn.Linear(
#     rs_model.fc.in_features,
#     num_classes
# )

# rs_model.load_state_dict(
#     checkpoint["model_state_dict"],
#     strict=True
# )

# start_epoch = checkpoint.get(
#     "epoch",
#     RESUME_EPOCH_IF_MISSING
# )

# best_val_accuracy = checkpoint.get(
#     "best_val_accuracy",
#     RESUME_BEST_ACC_IF_MISSING
# )


# rs_model = rs_model.to(device)

# rs_criterion = nn.CrossEntropyLoss()

# rs_optimizer = torch.optim.AdamW(
#     rs_model.parameters(),
#     lr=1e-3,
#     weight_decay=5e-4
# )

# rs_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#     rs_optimizer,
#     mode="max",       # moniotr val accuracy
#     factor=0.3,       # lr * 0.3
#     patience=2,       # if no increase drop lr
#     threshold=1e-4,
#     min_lr=1e-6
# )

# rs_model_filename = CHECKPOINT_PATH

# training(50,rs_model,rs_optimizer,rs_scheduler,rs_criterion,rs_model_filename)

ValueError: Checkpoint has 500 classes, but dataset has 20 classes.